In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['font.family'] = 'Malgun Gothic'
# plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['figure.figsize'] = 12, 6
plt.rcParams['font.size'] = 14
plt.rcParams['axes.unicode_minus'] = False

# 데이터 전처리 관련 ################################################
# 결측치 처리
from sklearn.impute import SimpleImputer
# 표준화
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
# 인코더
from sklearn.preprocessing import LabelEncoder
# 원핫 인코더
from sklearn.preprocessing import OneHotEncoder

# 학습 모델 성능 관련 #######################################
# 원하는 비율로 데이터를 나누기 위해
from sklearn.model_selection import train_test_split
# K-Fold 교차 검증
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold
# 학습 곡선
from sklearn.model_selection import learning_curve
# 하이퍼 파라미터 튜닝
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
optuna.logging.set_verbosity(optuna.logging.WARNING)

# 모델 성능 평가 ########################
# 회귀용
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import r2_score
# 분류용
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import roc_curve
from sklearn.metrics import auc
from sklearn.metrics import roc_auc_score
from sklearn.metrics import ConfusionMatrixDisplay

# 피처 선택 #########################
from sklearn.feature_selection import VarianceThreshold
from sklearn.feature_selection import RFE
from sklearn.inspection import permutation_importance

# 학습 모델 ########################
# 분류
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier 
from sklearn.naive_bayes import GaussianNB
from catboost import CatBoostClassifier

# 회귀
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.linear_model import ElasticNet
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import BayesianRidge
from catboost import CatBoostRegressor

# 결정트리를 시각화할 수 있는 라이브러리
from sklearn.tree import plot_tree

# 차원 축소
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.manifold import TSNE

# 연관 규칙 학습
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import fpgrowth
from mlxtend.frequent_patterns import association_rules

# 군집
from sklearn.cluster import KMeans
from sklearn.cluster import DBSCAN
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from sklearn.mixture import GaussianMixture
from sklearn.cluster import MeanShift, estimate_bandwidth

# 파이프라인
from sklearn.pipeline import Pipeline

# KDE를 그리기 위한 통계값을 구할 수 있는 함수
from scipy.stats import gaussian_kde

# 피어슨 상관 계수 (연속형 수치형 데이터 vs 연속형 수치형 데이터)
from scipy.stats import pearsonr
# 카이제곱 검증 (범주형 데이터 vs 범주형 데이터, 순위X)
from scipy.stats import chi2_contingency
# 스피어만 상관계수 (범주형 데이터 vs 범주형 데이터, 순위O)
from scipy.stats import spearmanr
# 포인트 이분 상관계수 (범주형 데이터 vs 연속형 수치형 데이터)
from scipy.stats import pointbiserialr

# 오버 샘플링
from imblearn.over_sampling import SMOTE

# 객체를 파일에 저장
import pickle

# 문장 데이터 관련
# 단어 토큰화
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
# 정규식 사용
import re
# 불용어 처리
from nltk.corpus import stopwords
# 어간 추출
from nltk.stem import PorterStemmer
# 표제어 추출
from nltk.stem import WordNetLemmatizer
# Bow와 TF-IDF를 위해..
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# 불필요한 경고 뜨지 않게
import warnings
warnings.filterwarnings('ignore')

### 실습 1 : 데이터 준비 및 BoW (단어 빈도)

In [2]:
# 빈도수를 계산할 단어들은 각각 하나의 문장으로 되어 있어야 한다.
# 예시 문장
corpus = [
    "Free entry win cash prize",
    "Free call to win free gift",
    "Call me when you are free"
]

In [3]:
# 학습 및 변환
count_v = CountVectorizer()
bow_matrix = count_v.fit_transform(corpus)
bow_matrix

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 16 stored elements and shape (3, 12)>

In [4]:
# 컬럼명 (단어 종류)
feature_names = count_v.get_feature_names_out()
feature_names

array(['are', 'call', 'cash', 'entry', 'free', 'gift', 'me', 'prize',
       'to', 'when', 'win', 'you'], dtype=object)

In [5]:
# 결과 확인을 하기 위해 데이터 프레임으로 생성한다.
df_bow = pd.DataFrame(bow_matrix.toarray(), columns=feature_names)
df_bow

,are,call,cash,entry,free,gift,me,prize,to,when,win,you
0,0,0,1,1,1,0,0,1,0,0,1,0
1,0,1,0,0,2,1,0,0,1,0,1,0
2,1,1,0,0,1,0,1,0,0,1,0,1


In [6]:
### 단어 사전
print(count_v.vocabulary_)

{'free': 4, 'entry': 3, 'win': 10, 'cash': 2, 'prize': 7, 'call': 1, 'to': 8, 'gift': 5, 'me': 6, 'when': 9, 'you': 11, 'are': 0}


### 실습 2 : TF-IDF (가중치 기반)

In [7]:
tfidf_v = TfidfVectorizer()
tfidf_matrix = tfidf_v.fit_transform(corpus)
tfidf_matrix

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 16 stored elements and shape (3, 12)>

In [8]:
# 컬럼명 (단어)
columns = tfidf_v.get_feature_names_out()
columns

array(['are', 'call', 'cash', 'entry', 'free', 'gift', 'me', 'prize',
       'to', 'when', 'win', 'you'], dtype=object)

In [9]:
df_tfidf = pd.DataFrame(tfidf_matrix.toarray(), columns=columns)
df_tfidf

,are,call,cash,entry,free,gift,me,prize,to,when,win,you
0,0.000000,0.000000,0.504611,0.504611,0.298032,0.000000,0.000000,0.504611,0.000000,0.000000,0.383770,0.000000
1,0.000000,0.356457,0.000000,0.000000,0.553642,0.468699,0.000000,0.000000,0.468699,0.000000,0.356457,0.000000
2,0.450504,0.342620,0.000000,0.000000,0.266075,0.000000,0.450504,0.000000,0.000000,0.450504,0.000000,0.450504


In [10]:
# (문장 순서 번호, 단어싀 순서 번호)
print(tfidf_matrix)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 16 stored elements and shape (3, 12)>
  Coords	Values
  (0, 4)	0.2980315863446099
  (0, 3)	0.5046113401371842
  (0, 10)	0.3837699307603192
  (0, 2)	0.5046113401371842
  (0, 7)	0.5046113401371842
  (1, 4)	0.5536419417527538
  (1, 10)	0.3564574014762071
  (1, 1)	0.3564574014762071
  (1, 8)	0.46869864635920433
  (1, 5)	0.46869864635920433
  (2, 4)	0.2660749625405929
  (2, 1)	0.3426199591918006
  (2, 6)	0.450504072643198
  (2, 9)	0.450504072643198
  (2, 11)	0.450504072643198
  (2, 0)	0.450504072643198


### 스팸 데이터에 적용해본다.

In [11]:
from ast import literal_eval

df = pd.read_csv('data/spam6.csv', encoding='latin-1')
df['final_tokens'] = df['final_tokens'].apply(literal_eval)
df

,label,text,tokenized_text,clean_text,clean_tokens,stemmed_tokens,final_tokens
0,ham,"Go until jurong point, crazy.. Available only ...","['Go', 'until', 'jurong', 'point', ',', 'crazy...",Go until jurong point crazy Available only in ...,"['Go', 'jurong', 'point', 'crazy', 'Available'...","['go', 'jurong', 'point', 'crazi', 'avail', 'b...","[go, jurong, point, crazi, avail, bugi, n, gre..."
1,ham,Ok lar... Joking wif u oni...,"['Ok', 'lar', '...', 'Joking', 'wif', 'u', 'on...",Ok lar Joking wif u oni,"['Ok', 'lar', 'Joking', 'wif', 'oni']","['ok', 'lar', 'joke', 'wif', 'oni']","[ok, lar, joke, wif, oni]"
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,"['Free', 'entry', 'in', '2', 'a', 'wkly', 'com...",Free entry in a wkly comp to win FA Cup final ...,"['entry', 'wkly', 'comp', 'win', 'FA', 'Cup', ...","['entri', 'wkli', 'comp', 'win', 'fa', 'cup', ...","[entri, wkli, comp, win, fa, cup, final, tkt, ..."
3,ham,U dun say so early hor... U c already then say...,"['U', 'dun', 'say', 'so', 'early', 'hor', '......",U dun say so early hor U c already then say,"['dun', 'say', 'early', 'hor', 'c', 'already',...","['dun', 'say', 'earli', 'hor', 'c', 'alreadi',...","[dun, say, earli, hor, c, alreadi, say]"
4,ham,"Nah I don't think he goes to usf, he lives aro...","['Nah', 'I', 'do', ""n't"", 'think', 'he', 'goes...",Nah I don t think he goes to usf he lives arou...,"['Nah', 'think', 'goes', 'usf', 'lives', 'arou...","['nah', 'think', 'goe', 'usf', 'live', 'around...","[nah, think, goe, usf, live, around, though]"
...,...,...,...,...,...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...,"['This', 'is', 'the', '2nd', 'time', 'we', 'ha...",This is the nd time we have tried contact u U ...,"['time', 'tried', 'contact', 'Pound', 'prize',...","['time', 'tri', 'contact', 'pound', 'prize', '...","[time, tri, contact, pound, prize, claim, easi..."
5568,ham,Will Ì_ b going to esplanade fr home?,"['Will', 'Ì_', 'b', 'going', 'to', 'esplanade'...",Will b going to esplanade fr home,"['b', 'going', 'esplanade', 'fr', 'home']","['b', 'go', 'esplanad', 'fr', 'home']","[b, go, esplanad, fr, home]"
5569,ham,"Pity, * was in mood for that. So...any other s...","['Pity', ',', '*', 'was', 'in', 'mood', 'for',...",Pity was in mood for that So any other suggest...,"['Pity', 'mood', 'suggestions']","['piti', 'mood', 'suggest']","[piti, mood, suggest]"
5570,ham,The guy did some bitching but I acted like i'd...,"['The', 'guy', 'did', 'some', 'bitching', 'but...",The guy did some bitching but I acted like i d...,"['guy', 'bitching', 'acted', 'like', 'interest...","['guy', 'bitch', 'act', 'like', 'interest', 'b...","[guy, bitch, act, like, interest, buy, someth,..."


In [12]:
# final tokens에 있는 단어들을 하나의 문자열로 만들어준다.
def make_final_text(word_list) :
    return ' '.join(word_list)

df['final_text'] = df['final_tokens'].apply(make_final_text)
df

,label,text,tokenized_text,clean_text,clean_tokens,stemmed_tokens,final_tokens,final_text
0,ham,"Go until jurong point, crazy.. Available only ...","['Go', 'until', 'jurong', 'point', ',', 'crazy...",Go until jurong point crazy Available only in ...,"['Go', 'jurong', 'point', 'crazy', 'Available'...","['go', 'jurong', 'point', 'crazi', 'avail', 'b...","[go, jurong, point, crazi, avail, bugi, n, gre...",go jurong point crazi avail bugi n great world...
1,ham,Ok lar... Joking wif u oni...,"['Ok', 'lar', '...', 'Joking', 'wif', 'u', 'on...",Ok lar Joking wif u oni,"['Ok', 'lar', 'Joking', 'wif', 'oni']","['ok', 'lar', 'joke', 'wif', 'oni']","[ok, lar, joke, wif, oni]",ok lar joke wif oni
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,"['Free', 'entry', 'in', '2', 'a', 'wkly', 'com...",Free entry in a wkly comp to win FA Cup final ...,"['entry', 'wkly', 'comp', 'win', 'FA', 'Cup', ...","['entri', 'wkli', 'comp', 'win', 'fa', 'cup', ...","[entri, wkli, comp, win, fa, cup, final, tkt, ...",entri wkli comp win fa cup final tkt may text ...
3,ham,U dun say so early hor... U c already then say...,"['U', 'dun', 'say', 'so', 'early', 'hor', '......",U dun say so early hor U c already then say,"['dun', 'say', 'early', 'hor', 'c', 'already',...","['dun', 'say', 'earli', 'hor', 'c', 'alreadi',...","[dun, say, earli, hor, c, alreadi, say]",dun say earli hor c alreadi say
4,ham,"Nah I don't think he goes to usf, he lives aro...","['Nah', 'I', 'do', ""n't"", 'think', 'he', 'goes...",Nah I don t think he goes to usf he lives arou...,"['Nah', 'think', 'goes', 'usf', 'lives', 'arou...","['nah', 'think', 'goe', 'usf', 'live', 'around...","[nah, think, goe, usf, live, around, though]",nah think goe usf live around though
...,...,...,...,...,...,...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...,"['This', 'is', 'the', '2nd', 'time', 'we', 'ha...",This is the nd time we have tried contact u U ...,"['time', 'tried', 'contact', 'Pound', 'prize',...","['time', 'tri', 'contact', 'pound', 'prize', '...","[time, tri, contact, pound, prize, claim, easi...",time tri contact pound prize claim easi call p...
5568,ham,Will Ì_ b going to esplanade fr home?,"['Will', 'Ì_', 'b', 'going', 'to', 'esplanade'...",Will b going to esplanade fr home,"['b', 'going', 'esplanade', 'fr', 'home']","['b', 'go', 'esplanad', 'fr', 'home']","[b, go, esplanad, fr, home]",b go esplanad fr home
5569,ham,"Pity, * was in mood for that. So...any other s...","['Pity', ',', '*', 'was', 'in', 'mood', 'for',...",Pity was in mood for that So any other suggest...,"['Pity', 'mood', 'suggestions']","['piti', 'mood', 'suggest']","[piti, mood, suggest]",piti mood suggest
5570,ham,The guy did some bitching but I acted like i'd...,"['The', 'guy', 'did', 'some', 'bitching', 'but...",The guy did some bitching but I acted like i d...,"['guy', 'bitching', 'acted', 'like', 'interest...","['guy', 'bitch', 'act', 'like', 'interest', 'b...","[guy, bitch, act, like, interest, buy, someth,...",guy bitch act like interest buy someth els nex...


In [13]:
# 빈 문자열("") 이나 공백만 있는 행 제거
# 전처리 후 아무 단어도 남지 않은 행들을 골래낸다(오류 방지)
df = df[df['final_text'].str.strip() != ""]
df

,label,text,tokenized_text,clean_text,clean_tokens,stemmed_tokens,final_tokens,final_text
0,ham,"Go until jurong point, crazy.. Available only ...","['Go', 'until', 'jurong', 'point', ',', 'crazy...",Go until jurong point crazy Available only in ...,"['Go', 'jurong', 'point', 'crazy', 'Available'...","['go', 'jurong', 'point', 'crazi', 'avail', 'b...","[go, jurong, point, crazi, avail, bugi, n, gre...",go jurong point crazi avail bugi n great world...
1,ham,Ok lar... Joking wif u oni...,"['Ok', 'lar', '...', 'Joking', 'wif', 'u', 'on...",Ok lar Joking wif u oni,"['Ok', 'lar', 'Joking', 'wif', 'oni']","['ok', 'lar', 'joke', 'wif', 'oni']","[ok, lar, joke, wif, oni]",ok lar joke wif oni
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,"['Free', 'entry', 'in', '2', 'a', 'wkly', 'com...",Free entry in a wkly comp to win FA Cup final ...,"['entry', 'wkly', 'comp', 'win', 'FA', 'Cup', ...","['entri', 'wkli', 'comp', 'win', 'fa', 'cup', ...","[entri, wkli, comp, win, fa, cup, final, tkt, ...",entri wkli comp win fa cup final tkt may text ...
3,ham,U dun say so early hor... U c already then say...,"['U', 'dun', 'say', 'so', 'early', 'hor', '......",U dun say so early hor U c already then say,"['dun', 'say', 'early', 'hor', 'c', 'already',...","['dun', 'say', 'earli', 'hor', 'c', 'alreadi',...","[dun, say, earli, hor, c, alreadi, say]",dun say earli hor c alreadi say
4,ham,"Nah I don't think he goes to usf, he lives aro...","['Nah', 'I', 'do', ""n't"", 'think', 'he', 'goes...",Nah I don t think he goes to usf he lives arou...,"['Nah', 'think', 'goes', 'usf', 'lives', 'arou...","['nah', 'think', 'goe', 'usf', 'live', 'around...","[nah, think, goe, usf, live, around, though]",nah think goe usf live around though
...,...,...,...,...,...,...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...,"['This', 'is', 'the', '2nd', 'time', 'we', 'ha...",This is the nd time we have tried contact u U ...,"['time', 'tried', 'contact', 'Pound', 'prize',...","['time', 'tri', 'contact', 'pound', 'prize', '...","[time, tri, contact, pound, prize, claim, easi...",time tri contact pound prize claim easi call p...
5568,ham,Will Ì_ b going to esplanade fr home?,"['Will', 'Ì_', 'b', 'going', 'to', 'esplanade'...",Will b going to esplanade fr home,"['b', 'going', 'esplanade', 'fr', 'home']","['b', 'go', 'esplanad', 'fr', 'home']","[b, go, esplanad, fr, home]",b go esplanad fr home
5569,ham,"Pity, * was in mood for that. So...any other s...","['Pity', ',', '*', 'was', 'in', 'mood', 'for',...",Pity was in mood for that So any other suggest...,"['Pity', 'mood', 'suggestions']","['piti', 'mood', 'suggest']","[piti, mood, suggest]",piti mood suggest
5570,ham,The guy did some bitching but I acted like i'd...,"['The', 'guy', 'did', 'some', 'bitching', 'but...",The guy did some bitching but I acted like i d...,"['guy', 'bitching', 'acted', 'like', 'interest...","['guy', 'bitch', 'act', 'like', 'interest', 'b...","[guy, bitch, act, like, interest, buy, someth,...",guy bitch act like interest buy someth els nex...


In [14]:
# 인덱스 초기화
df = df.reset_index(drop=True)
df

,label,text,tokenized_text,clean_text,clean_tokens,stemmed_tokens,final_tokens,final_text
0,ham,"Go until jurong point, crazy.. Available only ...","['Go', 'until', 'jurong', 'point', ',', 'crazy...",Go until jurong point crazy Available only in ...,"['Go', 'jurong', 'point', 'crazy', 'Available'...","['go', 'jurong', 'point', 'crazi', 'avail', 'b...","[go, jurong, point, crazi, avail, bugi, n, gre...",go jurong point crazi avail bugi n great world...
1,ham,Ok lar... Joking wif u oni...,"['Ok', 'lar', '...', 'Joking', 'wif', 'u', 'on...",Ok lar Joking wif u oni,"['Ok', 'lar', 'Joking', 'wif', 'oni']","['ok', 'lar', 'joke', 'wif', 'oni']","[ok, lar, joke, wif, oni]",ok lar joke wif oni
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,"['Free', 'entry', 'in', '2', 'a', 'wkly', 'com...",Free entry in a wkly comp to win FA Cup final ...,"['entry', 'wkly', 'comp', 'win', 'FA', 'Cup', ...","['entri', 'wkli', 'comp', 'win', 'fa', 'cup', ...","[entri, wkli, comp, win, fa, cup, final, tkt, ...",entri wkli comp win fa cup final tkt may text ...
3,ham,U dun say so early hor... U c already then say...,"['U', 'dun', 'say', 'so', 'early', 'hor', '......",U dun say so early hor U c already then say,"['dun', 'say', 'early', 'hor', 'c', 'already',...","['dun', 'say', 'earli', 'hor', 'c', 'alreadi',...","[dun, say, earli, hor, c, alreadi, say]",dun say earli hor c alreadi say
4,ham,"Nah I don't think he goes to usf, he lives aro...","['Nah', 'I', 'do', ""n't"", 'think', 'he', 'goes...",Nah I don t think he goes to usf he lives arou...,"['Nah', 'think', 'goes', 'usf', 'lives', 'arou...","['nah', 'think', 'goe', 'usf', 'live', 'around...","[nah, think, goe, usf, live, around, though]",nah think goe usf live around though
...,...,...,...,...,...,...,...,...
5549,spam,This is the 2nd time we have tried 2 contact u...,"['This', 'is', 'the', '2nd', 'time', 'we', 'ha...",This is the nd time we have tried contact u U ...,"['time', 'tried', 'contact', 'Pound', 'prize',...","['time', 'tri', 'contact', 'pound', 'prize', '...","[time, tri, contact, pound, prize, claim, easi...",time tri contact pound prize claim easi call p...
5550,ham,Will Ì_ b going to esplanade fr home?,"['Will', 'Ì_', 'b', 'going', 'to', 'esplanade'...",Will b going to esplanade fr home,"['b', 'going', 'esplanade', 'fr', 'home']","['b', 'go', 'esplanad', 'fr', 'home']","[b, go, esplanad, fr, home]",b go esplanad fr home
5551,ham,"Pity, * was in mood for that. So...any other s...","['Pity', ',', '*', 'was', 'in', 'mood', 'for',...",Pity was in mood for that So any other suggest...,"['Pity', 'mood', 'suggestions']","['piti', 'mood', 'suggest']","[piti, mood, suggest]",piti mood suggest
5552,ham,The guy did some bitching but I acted like i'd...,"['The', 'guy', 'did', 'some', 'bitching', 'but...",The guy did some bitching but I acted like i d...,"['guy', 'bitching', 'acted', 'like', 'interest...","['guy', 'bitch', 'act', 'like', 'interest', 'b...","[guy, bitch, act, like, interest, buy, someth,...",guy bitch act like interest buy someth els nex...


In [15]:
# Tfidf를 통해 단어 벡터를 만든다.
# token_pattern : 사용할 단어들의 정규시. 생략하면 두 글자 이상인 것만 사용한다. 
# 여기에서는 한 글자 이상도 포함하도록 정규식을 설정한다.
# max_features : 빈도수가 높은 5000개의 단어만 사용한다.
tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,
    token_pattern=r"(?u)\b\w+\b"
)

In [17]:
# 벡터화
x_tfidf = tfidf_vectorizer.fit_transform(df['final_text'])
print(x_tfidf)


<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 41447 stored elements and shape (5554, 5000)>
  Coords	Values
  (0, 1588)	0.14391240349865203
  (0, 2203)	0.3584902459180792
  (0, 3354)	0.2447435237060044
  (0, 648)	0.2803666193868821
  (0, 201)	0.2702769476995997
  (0, 377)	0.3028338514113685
  (0, 2839)	0.1934485174305396
  (0, 1655)	0.19834210133700467
  (0, 4919)	0.2402780811560157
  (0, 2301)	0.3028338514113685
  (0, 1023)	0.2135205500772008
  (0, 376)	0.3422117940621948
  (0, 495)	0.3028338514113685
  (0, 1624)	0.16645156316645815
  (0, 4831)	0.19797207473447448
  (1, 3052)	0.2850824267606541
  (1, 2329)	0.4204555469209925
  (1, 2174)	0.47696523736074664
  (1, 4879)	0.4444644970247425
  (1, 3075)	0.5629392651517343
  (2, 1125)	0.4093514832010647
  (2, 4903)	0.21667762301217885
  (2, 540)	0.2236703034773073
  (2, 4885)	0.16582737606738399
  (2, 1235)	0.5342256048927417
  :	:
  (5549, 2700)	0.34569805071408666
  (5549, 1031)	0.37401237896207745
  (5550, 1588)	0.2700132